# Обработка данных в Pandas

Практический ноутбук к модулю **«Обработка данных в Pandas»** программы **«Специалист по большим данным»**.

В ноутбуке используется единый учебный набор данных о котах:

- `cats_raw.csv` — основная «сырая» таблица;
- `Данные по котам для Pandas.xlsx` — Excel-файл с листами `cats_raw`, `breeds`, `visits`, `README`, `data_dictionary`.

Цель: пройти полный цикл работы с данными — **загрузка → диагностика → очистка → преобразование → анализ → объединение → агрегация → экспорт**.


## 1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print('Pandas:', pd.__version__)

Pandas: 3.0.3


## 2. Пути к файлам

Если ноутбук лежит в одной папке с данными, менять ничего не нужно.  
Для запуска в другой среде достаточно изменить `DATA_DIR`.


In [2]:
DATA_DIR = Path(".")
CSV_PATH = DATA_DIR / "cats_raw.csv"
XLSX_PATH = DATA_DIR / "Данные по котам для Pandas.xlsx"

print('CSV:', CSV_PATH)
print('XLSX:', XLSX_PATH)

CSV: cats_raw.csv
XLSX: Данные по котам для Pandas.xlsx


## 3. Чтение CSV

In [3]:
cats = pd.read_csv(CSV_PATH)
cats.head()

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
0,CAT001,Лео,Мейн-кун,кот,рыжий,2021-11-24,8.4,37.0,Томск,да,да,OWN001
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
2,CAT003,Злата,Шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,ДА,да,OWN011
3,CAT004,Мия,Сибирская,кошка,табби,2024-09-06,5.2,33.0,Санкт-Петербург,нет,да,OWN016
4,CAT005,Кира,Сиамская,кот,серый,2023-09-27,NaN,29.0,Екатеринбург,Нет,нет,OWN021


### Размер таблицы

In [4]:
cats.shape

(125, 12)

В исходном CSV должно быть **125 строк и 12 столбцов**.

Посмотрим названия полей.


In [5]:
cats.columns.tolist()

['cat_id',
 'name',
 'breed',
 'sex',
 'color',
 'birth_date',
 'weight_kg',
 'height_cm',
 'city',
 'vaccinated',
 'sterilized',
 'owner_id']

## 4. Чтение Excel

In [6]:
excel = pd.ExcelFile(XLSX_PATH)

excel.sheet_names

['README', 'cats_raw', 'breeds', 'visits', 'data_dictionary']

### Загружаем отдельные листы

In [7]:
cats_excel = pd.read_excel(XLSX_PATH, sheet_name='cats_raw')
breeds = pd.read_excel(XLSX_PATH, sheet_name='breeds')
visits = pd.read_excel(XLSX_PATH, sheet_name='visits')

print('cats_raw', cats_excel.shape)
print('breeds', breeds.shape)
print('visits', visits.shape)

cats_raw (125, 12)
breeds (12, 6)
visits (210, 7)


## 5. Первичное знакомство с DataFrame

In [8]:
cats.head(n=7)

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
0,CAT001,Лео,Мейн-кун,кот,рыжий,2021-11-24,8.4,37.0,Томск,да,да,OWN001
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
2,CAT003,Злата,Шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,ДА,да,OWN011
3,CAT004,Мия,Сибирская,кошка,табби,2024-09-06,5.2,33.0,Санкт-Петербург,нет,да,OWN016
4,CAT005,Кира,Сиамская,кот,серый,2023-09-27,NaN,29.0,Екатеринбург,Нет,нет,OWN021
5,CAT006,Плюша,Канадский сфинкс,кошка,серебристый,2018-09-13,4.4,30.0,Томск,НЕТ,да,OWN026
6,CAT007,Ириска,Бенгальская,кот,белый,2018-05-07,"4,9",32.0,Новосибирск,да,нет,OWN031


In [9]:
cats.tail(n=7)

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
118,CAT119,Тиша,Русская голубая,кот,черепаховый,2024-12-26,4.5,30.0,Томск,Нет,да,OWN010
119,CAT120,Том,Норвежская лесная,кошка,голубой,2021-03-12,6.9,37.0,Москва,НЕТ,нет,OWN015
120,CAT003,Злата,Шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,ДА,да,OWN011
121,CAT019,Кира,Бенгальская,кот,рыжий,2024-04-12,4.4,30.0,Новосибирск,да,да,OWN008
122,CAT055,Мия,Бенгальская,кот,рыжий,2020-01-04,6.2,33.0,Томск,да,нет,OWN022
123,CAT078,Тиша,Канадский сфинкс,кошка,табби,2017-03-20,5.2,33.0,Томск,НЕТ,да,OWN054
124,CAT104,Буся,Персидская,кошка,серебристый,2016-07-15,4.3,29.0,Красноярск,Да,да,OWN018


### Структура таблицы

In [10]:
cats.info()

<class 'pandas.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   cat_id      125 non-null    str    
 1   name        125 non-null    str    
 2   breed       125 non-null    str    
 3   sex         125 non-null    str    
 4   color       118 non-null    str    
 5   birth_date  125 non-null    str    
 6   weight_kg   117 non-null    str    
 7   height_cm   123 non-null    float64
 8   city        125 non-null    str    
 9   vaccinated  125 non-null    str    
 10  sterilized  125 non-null    str    
 11  owner_id    120 non-null    str    
dtypes: float64(1), str(11)
memory usage: 25.2 KB


### Типы данных

In [11]:
cats.dtypes

cat_id            str
name              str
breed             str
sex               str
color             str
birth_date        str
weight_kg         str
height_cm     float64
city              str
vaccinated        str
sterilized        str
owner_id          str
dtype: object

### Описательная статистика

In [12]:
cats.describe()

,height_cm
count,123.000000
mean,31.886179
std,3.383187
min,25.000000
25%,29.500000
50%,32.000000
75%,34.000000
max,42.000000


### Статистика по строковым столбцам

In [13]:
cats.describe(include='string')

,cat_id,name,breed,sex,color,birth_date,weight_kg,city,vaccinated,sterilized,owner_id
count,125,125,125,125,118,125,117,125,125,125,120
unique,120,29,17,2,10,119,54,8,6,2,81
top,CAT003,Кира,Бенгальская,кот,рыжий,2024-04-01,4.4,Томск,да,да,OWN011
freq,2,10,12,63,16,2,5,21,22,76,3


### Количество уникальных значений

In [14]:
cats.nunique().sort_values()

sex             2
sterilized      2
vaccinated      6
city            8
color          10
breed          17
height_cm      17
name           29
weight_kg      54
owner_id       81
birth_date    119
cat_id        120
dtype: int64

## 6. Выбор столбцов

In [15]:
cats['name'].head()

0      Лео
1      Лео
2    Злата
3      Мия
4     Кира
Name: name, dtype: str

In [16]:
cats[['name', 'breed', 'weight_kg', 'city']].head()

,name,breed,weight_kg,city
0,Лео,Мейн-кун,8.4,Томск
1,Лео,Британская короткошёрстная,4.4,Москва
2,Злата,Шотландская вислоухая,6.3,Москва
3,Мия,Сибирская,5.2,Санкт-Петербург
4,Кира,Сиамская,NaN,Екатеринбург


## 7. loc и iloc

In [17]:
cats.loc[0:4]

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
0,CAT001,Лео,Мейн-кун,кот,рыжий,2021-11-24,8.4,37.0,Томск,да,да,OWN001
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
2,CAT003,Злата,Шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,ДА,да,OWN011
3,CAT004,Мия,Сибирская,кошка,табби,2024-09-06,5.2,33.0,Санкт-Петербург,нет,да,OWN016
4,CAT005,Кира,Сиамская,кот,серый,2023-09-27,NaN,29.0,Екатеринбург,Нет,нет,OWN021


In [18]:
cats.iloc[0:4]

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
0,CAT001,Лео,Мейн-кун,кот,рыжий,2021-11-24,8.4,37.0,Томск,да,да,OWN001
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
2,CAT003,Злата,Шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,ДА,да,OWN011
3,CAT004,Мия,Сибирская,кошка,табби,2024-09-06,5.2,33.0,Санкт-Петербург,нет,да,OWN016


## 8. Фильтрация строк

In [19]:
cats[cats['city'] == 'Москва'].head()

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
2,CAT003,Злата,Шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,ДА,да,OWN011
17,CAT018,Персик,Канадский сфинкс,кошка,черепаховый,31.02.2021,4.0,30.0,Москва,НЕТ,да,OWN003
21,CAT022,Граф,Абиссинская,кошка,кремовый,2018-03-13,5.3,30.0,Москва,нет,нет,OWN023
27,CAT028,Граф,Сибирская,кошка,черепаховый,2022-08-31,7.1,35.0,Москва,нет,да,OWN053


### Несколько условий

In [20]:
moscow_female = cats[
    (cats['city'] == 'Москва') &
    (cats['sex'] == 'кошка')
]

moscow_female.head(10)

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
17,CAT018,Персик,Канадский сфинкс,кошка,черепаховый,31.02.2021,4.0,30.0,Москва,НЕТ,да,OWN003
21,CAT022,Граф,Абиссинская,кошка,кремовый,2018-03-13,5.3,30.0,Москва,нет,нет,OWN023
27,CAT028,Граф,Сибирская,кошка,черепаховый,2022-08-31,7.1,35.0,Москва,нет,да,OWN053
31,CAT032,Алиса,Персидская,кошка,колор-пойнт,2023-08-09,нет данных,29.0,Москва,Да,нет,OWN073
35,CAT036,Кузя,Норвежская лесная,кошка,голубой,2022-01-19,5.0,31.0,Москва,НЕТ,да,OWN010
41,CAT042,Симба,Канадский сфинкс,кошка,рыжий,2020-06-01,3.5,26.0,Москва,НЕТ,нет,OWN040
43,CAT044,Оскар,Персидская,кошка,чёрный,2016-10-11,5.8,35.0,Москва,Да,да,OWN050
45,CAT046,Снежок,Абиссинская,кошка,NaN,2022-09-27,2.9,27.0,Москва,нет,да,OWN060
51,CAT052,Снежок,Сибирская,кошка,серебристый,2017-06-04,6.1,33.0,Москва,нет,нет,OWN007


### isin()

In [21]:
cats[
    (cats['city'] == 'Москва') |
    (cats['city'] == 'Новосибирск') |
    (cats['city'] == 'Томск')
].head()

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
0,CAT001,Лео,Мейн-кун,кот,рыжий,2021-11-24,8.4,37.0,Томск,да,да,OWN001
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
2,CAT003,Злата,Шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,ДА,да,OWN011
5,CAT006,Плюша,Канадский сфинкс,кошка,серебристый,2018-09-13,4.4,30.0,Томск,НЕТ,да,OWN026
6,CAT007,Ириска,Бенгальская,кот,белый,2018-05-07,"4,9",32.0,Новосибирск,да,нет,OWN031


In [22]:
selected_cities = ['Москва', 'Новосибирск', 'Томск']

cats[cats['city'].isin(selected_cities)].head()

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
0,CAT001,Лео,Мейн-кун,кот,рыжий,2021-11-24,8.4,37.0,Томск,да,да,OWN001
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
2,CAT003,Злата,Шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,ДА,да,OWN011
5,CAT006,Плюша,Канадский сфинкс,кошка,серебристый,2018-09-13,4.4,30.0,Томск,НЕТ,да,OWN026
6,CAT007,Ириска,Бенгальская,кот,белый,2018-05-07,"4,9",32.0,Новосибирск,да,нет,OWN031


### query()

In [23]:
cats.query("city == 'Москва' and sex == 'кошка'").head()

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,Да,нет,OWN006
17,CAT018,Персик,Канадский сфинкс,кошка,черепаховый,31.02.2021,4.0,30.0,Москва,НЕТ,да,OWN003
21,CAT022,Граф,Абиссинская,кошка,кремовый,2018-03-13,5.3,30.0,Москва,нет,нет,OWN023
27,CAT028,Граф,Сибирская,кошка,черепаховый,2022-08-31,7.1,35.0,Москва,нет,да,OWN053
31,CAT032,Алиса,Персидская,кошка,колор-пойнт,2023-08-09,нет данных,29.0,Москва,Да,нет,OWN073


## 9. Сортировка

In [24]:
cats.sort_values('city')

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
4,CAT005,Кира,Сиамская,кот,серый,2023-09-27,NaN,29.0,Екатеринбург,Нет,нет,OWN021
11,CAT012,Марта,Норвежская лесная,кошка,белый,2023-04-07,7.8,39.0,Екатеринбург,НЕТ,нет,OWN056
20,CAT021,Кира,Рэгдолл,кот,черепаховый,2024-05-17,5.0,32.0,Екатеринбург,ДА,да,OWN018
22,CAT023,Ириска,Русская голубая,кот,белый,2023-10-27,"6,2",33.0,Екатеринбург,Нет,да,OWN028
25,CAT026,Соня,британская короткошёрстная,кошка,колор-пойнт,2020-12-01,3.9,27.0,Екатеринбург,Да,да,OWN043
...,...,...,...,...,...,...,...,...,...,...,...,...
112,CAT113,Луна,Сиамская,кот,серый,2020-09-11,4.3,30.0,Томск,Нет,да,NaN
117,CAT118,Марс,Абиссинская,кошка,серый,2024-04-06,"4,5",28.0,Томск,нет,да,OWN005
118,CAT119,Тиша,Русская голубая,кот,черепаховый,2024-12-26,4.5,30.0,Томск,Нет,да,OWN010
122,CAT055,Мия,Бенгальская,кот,рыжий,2020-01-04,6.2,33.0,Томск,да,нет,OWN022


In [25]:
cats.sort_values(['city', 'weight_kg'], ascending=[True, False])

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
48,CAT049,Алиса,Мейн-кун,кот,серебристый,2025-03-09,9.1,39.0,Екатеринбург,да,да,OWN075
11,CAT012,Марта,Норвежская лесная,кошка,белый,2023-04-07,7.8,39.0,Екатеринбург,НЕТ,нет,OWN056
83,CAT084,Кузя,Норвежская лесная,кошка,колор-пойнт,2020-01-08,7.5,37.0,Екатеринбург,НЕТ,да,OWN001
42,CAT043,Граф,Бенгальская,кот,колор-пойнт,2021-12-02,7.3,38.0,Екатеринбург,да,да,OWN045
80,CAT081,Оскар,Рэгдолл,кот,серый,2019-06-11,7.3,38.0,Екатеринбург,ДА,да,OWN069
...,...,...,...,...,...,...,...,...,...,...,...,...
111,CAT112,Тиша,Сибирская,кошка,черепаховый,2024-04-29,4.5,30.0,Томск,нет,нет,OWN058
118,CAT119,Тиша,Русская голубая,кот,черепаховый,2024-12-26,4.5,30.0,Томск,Нет,да,OWN010
5,CAT006,Плюша,Канадский сфинкс,кошка,серебристый,2018-09-13,4.4,30.0,Томск,НЕТ,да,OWN026
112,CAT113,Луна,Сиамская,кот,серый,2020-09-11,4.3,30.0,Томск,Нет,да,NaN


# Часть II. Очистка данных

Далее создадим отдельную копию таблицы, чтобы исходные данные всегда оставались доступными.


In [26]:
cats_clean = cats.copy()
cats_clean.shape

(125, 12)

## 10. Поиск пропусков

In [27]:
cats_clean.isna().sum()

cat_id        0
name          0
breed         0
sex           0
color         7
birth_date    0
weight_kg     8
height_cm     2
city          0
vaccinated    0
sterilized    0
owner_id      5
dtype: int64

### Доля пропусков в процентах

In [28]:
cats_clean.isna().mean().mul(100).round().sort_values(ascending=False)

color         6.0
weight_kg     6.0
owner_id      4.0
height_cm     2.0
cat_id        0.0
name          0.0
breed         0.0
sex           0.0
birth_date    0.0
city          0.0
vaccinated    0.0
sterilized    0.0
dtype: float64

### Строки, в которых отсутствует вес

In [29]:
cats_clean[cats_clean['weight_kg'].isna()]

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id
4,CAT005,Кира,Сиамская,кот,серый,2023-09-27,NaN,29.0,Екатеринбург,Нет,нет,OWN021
19,CAT020,Соня,Персидская,кошка,кремовый,2023-12-18,NaN,30.0,Санкт-Петербург,Да,нет,OWN013
28,CAT029,Ириска,Сиамская,кот,колор-пойнт,2024-09-22,NaN,28.0,Санкт-Петербург,Нет,да,OWN058
46,CAT047,Злата,Русская голубая,кот,белый,2024-01-09,NaN,29.0,Красноярск,Нет,нет,OWN065
58,CAT059,Лео,Русская голубая,кот,голубой,2021-04-28,NaN,33.0,Новосибирск,Нет,да,OWN042
72,CAT073,Мия,Мейн-кун,кот,белый,2019-03-20,NaN,34.0,Санкт-Петербург,да,да,OWN029
91,CAT092,Злата,Персидская,кошка,чёрный,2024-07-23,NaN,31.0,Казань,Да,нет,OWN041
114,CAT115,Симба,Бенгальская,кот,чёрный,2020-01-30,NaN,30.0,Екатеринбург,да,нет,OWN073


## 11. Дубликаты

In [30]:
cats_clean.duplicated().sum()

np.int64(5)

### Удаляем дубликаты

In [31]:
cats_clean = cats_clean.drop_duplicates().copy()
cats_clean.shape

(120, 12)

## 12. Очистка текстовых значений

In [32]:
cats_clean['breed'].unique()

<ArrowStringArray>
[                  'Мейн-кун', 'Британская короткошёрстная',
      'Шотландская вислоухая',                  'Сибирская',
                   'Сиамская',           'Канадский сфинкс',
                'Бенгальская',                 'Персидская',
                 '  Рэгдолл ',                'Абиссинская',
            'Русская голубая',          'Норвежская лесная',
                    'Рэгдолл', 'британская короткошёрстная',
              '  Персидская ',                'абиссинская',
                '  Мейн-кун ']
Length: 17, dtype: str

### Убираем лишние пробелы

In [33]:
cats_clean['breed'] = cats_clean['breed'].str.strip()

cats_clean['breed'].unique()

<ArrowStringArray>
[                  'Мейн-кун', 'Британская короткошёрстная',
      'Шотландская вислоухая',                  'Сибирская',
                   'Сиамская',           'Канадский сфинкс',
                'Бенгальская',                 'Персидская',
                    'Рэгдолл',                'Абиссинская',
            'Русская голубая',          'Норвежская лесная',
 'британская короткошёрстная',                'абиссинская']
Length: 14, dtype: str

### Нормализуем регистр пород

In [34]:
cats_clean['breed'] = (
    cats_clean['breed']
    .str.strip()
    .str.lower()
)

cats_clean['breed'].value_counts()

breed
мейн-кун                      10
британская короткошёрстная    10
шотландская вислоухая         10
сибирская                     10
сиамская                      10
канадский сфинкс              10
бенгальская                   10
персидская                    10
рэгдолл                       10
абиссинская                   10
русская голубая               10
норвежская лесная             10
Name: count, dtype: int64

## 13. Нормализация признака vaccinated

In [35]:
cats_clean['vaccinated'] = (
    cats_clean['vaccinated']
    .str.strip()
    .str.lower()
)

cats_clean['vaccinated'].value_counts(dropna=False)

vaccinated
да     60
нет    60
Name: count, dtype: int64

## 14. Очистка веса

In [36]:
cats_clean['weight_kg'].head(20)

0     8.4
1     4.4
2     6.3
3     5.2
4     NaN
5     4.4
6     4,9
7     3.3
8     5.3
9     3.6
10    4.7
11    7.8
12    5.4
13    6.0
14    3.8
15    4.1
16    3.4
17    4.0
18    4.4
19    NaN
Name: weight_kg, dtype: str

In [37]:
cats_clean['weight_kg'] = (
    cats_clean['weight_kg']
    .astype('string')
    .str.strip()
    .str.replace(",", ".", regex=False)
)

cats_clean['weight_kg'].head(20)


0      8.4
1      4.4
2      6.3
3      5.2
4     <NA>
5      4.4
6      4.9
7      3.3
8      5.3
9      3.6
10     4.7
11     7.8
12     5.4
13     6.0
14     3.8
15     4.1
16     3.4
17     4.0
18     4.4
19    <NA>
Name: weight_kg, dtype: string

In [38]:
cats_clean['weight_kg'] = pd.to_numeric(
    cats_clean['weight_kg'],
    errors='coerce'
)

cats_clean['weight_kg'].head(20)

0      8.4
1      4.4
2      6.3
3      5.2
4     <NA>
5      4.4
6      4.9
7      3.3
8      5.3
9      3.6
10     4.7
11     7.8
12     5.4
13     6.0
14     3.8
15     4.1
16     3.4
17     4.0
18     4.4
19    <NA>
Name: weight_kg, dtype: Float64

In [39]:
cats_clean['weight_kg'].dtype

Float64Dtype()

In [40]:
cats_clean['weight_kg'].isna().sum()

np.int64(12)

In [41]:
median_weight = cats_clean['weight_kg'].median()

cats_clean['weight_kg'] = (
    cats_clean['weight_kg']
    .fillna(median_weight)
)

median_weight

np.float64(5.15)

## 15. Преобразование дат

In [42]:
cats_clean['birth_date'].head(20)

0     2021-11-24
1     2016-09-04
2     2024-04-01
3     2024-09-06
4     2023-09-27
5     2018-09-13
6     2018-05-07
7     2020-10-22
8     2016-08-13
9     2019-06-26
10    2019-03-23
11    2023-04-07
12    2022-03-30
13    2024-02-11
14    2020-12-08
15    2022-03-31
16    2023-12-06
17    31.02.2021
18    2024-04-12
19    2023-12-18
Name: birth_date, dtype: str

In [43]:
cats_clean['birth_date'] = pd.to_datetime(
    cats_clean['birth_date'],
    errors = 'coerce'
)

In [44]:
cats_clean['birth_date'].head(20)

0    2021-11-24
1    2016-09-04
2    2024-04-01
3    2024-09-06
4    2023-09-27
5    2018-09-13
6    2018-05-07
7    2020-10-22
8    2016-08-13
9    2019-06-26
10   2019-03-23
11   2023-04-07
12   2022-03-30
13   2024-02-11
14   2020-12-08
15   2022-03-31
16   2023-12-06
17          NaT
18   2024-04-12
19   2023-12-18
Name: birth_date, dtype: datetime64[us]

## 16. Создание новых признаков

In [45]:
cats_clean['weight_g'] = cats_clean['weight_kg'] * 1000
cats_clean[['weight_kg','weight_g']]

,weight_kg,weight_g
0,8.4,8400.0
1,4.4,4400.0
2,6.3,6300.0
3,5.2,5200.0
4,5.15,5150.0
...,...,...
115,5.5,5500.0
116,6.1,6100.0
117,4.5,4500.0
118,4.5,4500.0


In [46]:
cats_clean['is_large'] = cats_clean['weight_kg'] >= 6

## 17. map()

In [47]:
cats_clean['sex_code'] = cats_clean['sex'].map({
    'кот':'M',
    'кошка':'F'
})

cats_clean[['sex', 'sex_code']]

,sex,sex_code
0,кот,M
1,кошка,F
2,кот,M
3,кошка,F
4,кот,M
...,...,...
115,кошка,F
116,кот,M
117,кошка,F
118,кот,M


## 18. apply()

In [48]:
def weight_category(weight):
    if pd.isna(weight):
        return 'нет данных'
    if weight < 4:
        return 'легкий'
    elif weight < 6:
        return 'средний'
    else:
        return 'крупный'

cats_clean['weight_category'] = cats_clean['weight_kg'].apply(weight_category)

cats_clean['weight_category'].value_counts()

weight_category
средний    66
крупный    35
легкий     19
Name: count, dtype: int64

## 19. Векторизация вместо цикла

Для простых операций предпочтительнее использовать векторизованные выражения Pandas.


In [49]:
cats_clean['height_m'] = cats_clean['height_cm'] / 100
cats_clean[['height_m', 'height_cm']]

,height_m,height_cm
0,0.37,37.0
1,0.31,31.0
2,0.35,35.0
3,0.33,33.0
4,0.29,29.0
...,...,...
115,0.32,32.0
116,0.35,35.0
117,0.28,28.0
118,0.30,30.0


# Часть III. Анализ данных

## 20. value_counts()

In [50]:
cats_clean['city'].value_counts()

city
Москва             20
Томск              19
Екатеринбург       16
Новосибирск        16
Санкт-Петербург    15
Красноярск         13
Казань             12
Омск                9
Name: count, dtype: int64

### Нормализованные доли

In [51]:
cats_clean['city'].value_counts(normalize=True).mul(100).round(2)

city
Москва             16.67
Томск              15.83
Екатеринбург       13.33
Новосибирск        13.33
Санкт-Петербург    12.50
Красноярск         10.83
Казань             10.00
Омск                7.50
Name: proportion, dtype: float64

## 21. Базовые агрегаты

In [52]:
cats_clean['weight_kg'].agg([
    'count', 'mean', 'median', 'min', 'max'
]).round(2)

count     120.00
mean        5.37
median      5.15
min         2.90
max         9.20
Name: weight_kg, dtype: float64

## 22. GroupBy

In [53]:
cats_clean.groupby('city')['weight_kg'].median().sort_values(ascending=False)

city
Екатеринбург       5.425
Омск                 5.4
Томск                5.3
Красноярск          5.15
Москва              5.15
Новосибирск         5.15
Санкт-Петербург     5.15
Казань              4.65
Name: weight_kg, dtype: Float64

In [54]:
print(0.1 + 0.1 == 0.2)
print(0.1 + 0.1 + 0.1 == 0.3)

True
False


### Несколько показателей

In [55]:
city_stats = (
    cats_clean
    .groupby('city')
    .agg(
        cats_count = ('cat_id', 'count'),
        avg_weight = ('weight_kg', 'mean'),
        median_weight = ('weight_kg', 'median'),
        avg_height = ('height_cm', 'mean')
    )
    .round(2)
    .sort_values('cats_count', ascending=False)
)
city_stats

,cats_count,avg_weight,median_weight,avg_height
city,,,,
Москва,20,5.24,5.15,31.58
Томск,19,5.6,5.3,32.68
Екатеринбург,16,5.88,5.43,32.62
Новосибирск,16,5.42,5.15,31.67
Санкт-Петербург,15,5.08,5.15,31.60
Красноярск,13,5.28,5.15,31.23
Казань,12,4.75,4.65,30.58
Омск,9,5.64,5.4,33.00


## 23. GroupBy по нескольким полям

In [56]:
(
    cats_clean
    .groupby(['city','sex'])
    .agg(
        cats_count = ('cat_id', 'count'),
        avg_weight = ('weight_kg', 'mean')
    )
    .round(2)
)

cats_count  avg_weight
city            sex                          
Екатеринбург    кот            11        5.95
                кошка           5         5.7
Казань          кот             4         4.1
                кошка           8        5.08
Красноярск      кот             5        4.93
                кошка           8        5.49
Москва          кот             3        6.17
                кошка          17        5.07
Новосибирск     кот            10        5.42
                кошка           6        5.41
Омск            кот             6        5.53
                кошка           3        5.87
Санкт-Петербург кот             9        5.68
                кошка           6        4.18
Томск           кот            12        5.81
                кошка           7        5.24

## 24. pivot_table()

In [57]:
city_sex_pivot = pd.pivot_table(
    cats_clean,
    index = 'city',
    columns = 'sex',
    values = 'weight_kg',
    aggfunc = ['count', 'median'],
    fill_value = 0
)

city_sex_pivot

count       median       
sex               кот кошка    кот  кошка
city                                     
Екатеринбург       11     5    5.7    4.8
Казань              4     8    3.8   4.75
Красноярск          5     8    5.1  5.325
Москва              3    17    6.1    5.1
Новосибирск        10     6  5.025  5.325
Омск                6     3    5.1    5.9
Санкт-Петербург     9     6   5.15   4.05
Томск              12     7   5.35    5.2

## 25. Справочник пород

In [58]:
breeds.head()

,breed,origin,coat_type,typical_weight_min_kg,typical_weight_max_kg,typical_lifespan_years
0,Мейн-кун,США,полудлинная,5.0,9.0,13
1,Британская короткошёрстная,Великобритания,короткая,4.0,8.0,14
2,Шотландская вислоухая,Великобритания,короткая,3.0,6.0,14
3,Сибирская,Россия,полудлинная,4.0,8.0,15
4,Сиамская,Таиланд,короткая,3.0,5.5,16


In [59]:
breeds.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   breed                   12 non-null     str    
 1   origin                  12 non-null     str    
 2   coat_type               12 non-null     str    
 3   typical_weight_min_kg   12 non-null     float64
 4   typical_weight_max_kg   12 non-null     float64
 5   typical_lifespan_years  12 non-null     int64  
dtypes: float64(2), int64(1), str(3)
memory usage: 1.4 KB


In [60]:
breeds_clean = breeds.copy()

breeds_clean['breed_key'] = (
    breeds_clean['breed']
    .str.strip()
    .str.lower()
)

cats_clean['breed_key'] = (
    cats_clean['breed']
    .str.strip()
    .str.lower()
)

## 26. merge(): cats + breeds

In [61]:
cats_with_breeds = cats_clean.merge(
    breeds_clean,
    on = 'breed_key',
    how='left',
    validate='many_to_one',
    suffixes=("", "_ref")
)

cats_with_breeds

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,...,sex_code,weight_category,height_m,breed_key,breed_ref,origin,coat_type,typical_weight_min_kg,typical_weight_max_kg,typical_lifespan_years
0,CAT001,Лео,мейн-кун,кот,рыжий,2021-11-24,8.4,37.0,Томск,да,...,M,крупный,0.37,мейн-кун,Мейн-кун,США,полудлинная,5.0,9.0,13
1,CAT002,Лео,британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,да,...,F,средний,0.31,британская короткошёрстная,Британская короткошёрстная,Великобритания,короткая,4.0,8.0,14
2,CAT003,Злата,шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,да,...,M,крупный,0.35,шотландская вислоухая,Шотландская вислоухая,Великобритания,короткая,3.0,6.0,14
3,CAT004,Мия,сибирская,кошка,табби,2024-09-06,5.2,33.0,Санкт-Петербург,нет,...,F,средний,0.33,сибирская,Сибирская,Россия,полудлинная,4.0,8.0,15
4,CAT005,Кира,сиамская,кот,серый,2023-09-27,5.15,29.0,Екатеринбург,нет,...,M,средний,0.29,сиамская,Сиамская,Таиланд,короткая,3.0,5.5,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,CAT116,Лео,персидская,кошка,белый,2021-05-09,5.5,32.0,Новосибирск,да,...,F,средний,0.32,персидская,Персидская,Иран,длинная,3.0,6.0,14
116,CAT117,Лео,рэгдолл,кот,NaN,2023-07-22,6.1,35.0,Москва,да,...,M,крупный,0.35,рэгдолл,Рэгдолл,США,полудлинная,4.5,9.0,15
117,CAT118,Марс,абиссинская,кошка,серый,2024-04-06,4.5,28.0,Томск,нет,...,F,средний,0.28,абиссинская,Абиссинская,Эфиопия,короткая,3.0,5.5,15
118,CAT119,Тиша,русская голубая,кот,черепаховый,2024-12-26,4.5,30.0,Томск,нет,...,M,средний,0.30,русская голубая,Русская голубая,Россия,короткая,3.0,6.0,16


## 27. Таблица ветеринарных визитов

In [62]:
visits.head()

,visit_id,cat_id,visit_date,reason,clinic,cost_rub,follow_up_days
0,VIS0001,CAT001,2026-08-09,профилактический осмотр,Айболит,NaN,14.0
1,VIS0002,CAT001,2026-06-17,анализ крови,Белый клык,1641.0,NaN
2,VIS0003,CAT001,2026-04-25,дерматология,Доктор Вет,2182.0,NaN
3,VIS0004,CAT002,2026-07-23,вакцинация,Кот и пёс,1479.0,NaN
4,VIS0005,CAT003,2026-07-06,анализ крови,Белый клык,1858.0,NaN


In [63]:
visits.info()

<class 'pandas.DataFrame'>
RangeIndex: 210 entries, 0 to 209
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   visit_id        210 non-null    str           
 1   cat_id          210 non-null    str           
 2   visit_date      210 non-null    datetime64[us]
 3   reason          210 non-null    str           
 4   clinic          210 non-null    str           
 5   cost_rub        202 non-null    float64       
 6   follow_up_days  66 non-null     float64       
dtypes: datetime64[us](1), float64(2), str(4)
memory usage: 23.6 KB


In [64]:
visits_clean = visits.copy()

visits_clean['visit_date'] = pd.to_datetime(
    visits_clean['visit_date'],
    errors = 'coerce'
)

visits_clean['visit_date'].min(), visits_clean['visit_date'].max()

(Timestamp('2025-06-08 00:00:00'), Timestamp('2026-08-09 00:00:00'))

## 28. merge(): cats + visits

In [65]:
cats_visits = cats_clean.merge(
    visits_clean,
    on = 'cat_id',
    how = 'left',
    validate = 'one_to_many'
)

print('Коты:', cats_clean.shape)
print('Визиты:', visits_clean.shape)
print('После JOIN:', cats_visits.shape)

Коты: (120, 18)
Визиты: (210, 7)
После JOIN: (210, 24)


## 29. Количество визитов на кота

In [66]:
visits_per_cat = (
    visits_clean
    .groupby('cat_id')
    .agg(
        visits_count = ('visit_id', 'count'),
        total_cost = ('cost_rub', 'sum'),
        avg_cost = ('cost_rub', 'mean'),
        last_visit = ('visit_date', 'max')
    )
)

visits_per_cat.head(10)

,visits_count,total_cost,avg_cost,last_visit
cat_id,,,,
CAT001,3,3823.0,1911.5,2026-08-09
CAT002,1,1479.0,1479.0,2026-07-23
CAT003,1,1858.0,1858.0,2026-07-06
CAT004,2,5015.0,2507.5,2026-06-19
CAT005,3,9471.0,3157.0,2026-06-02
CAT006,1,2995.0,2995.0,2026-05-16
CAT007,2,7289.0,3644.5,2026-04-29
CAT008,1,3753.0,3753.0,2026-04-12
CAT009,3,14019.0,4673.0,2026-03-26


### Добавляем агрегаты визитов к котам

In [67]:
cats_analytics = cats_with_breeds.merge(
    visits_per_cat,
    on = 'cat_id',
    how = 'left',
    validate = 'one_to_one'
)

In [68]:
cats_analytics.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 28 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   cat_id                  120 non-null    str           
 1   name                    120 non-null    str           
 2   breed                   120 non-null    str           
 3   sex                     120 non-null    str           
 4   color                   113 non-null    str           
 5   birth_date              117 non-null    datetime64[us]
 6   weight_kg               120 non-null    Float64       
 7   height_cm               118 non-null    float64       
 8   city                    120 non-null    str           
 9   vaccinated              120 non-null    str           
 10  sterilized              120 non-null    str           
 11  owner_id                115 non-null    str           
 12  weight_g                120 non-null    Float64       
 13  i

## 30. Анализ расходов по городам

In [69]:
cats_city_analytics = (
    cats_analytics
    .groupby('city')
    .agg(
        visits = ('visits_count', 'sum'),
        total_cost = ('total_cost','sum'),
        max_cost = ('total_cost','max'),
        min_cost = ('total_cost','min')
    )
)
cats_city_analytics['avg_cost'] = (
    cats_city_analytics['total_cost'] / cats_city_analytics['visits']
).round(2)

In [70]:
cats_city_analytics

,visits,total_cost,max_cost,min_cost,avg_cost
city,,,,,
Екатеринбург,32,156341.0,27663.0,0.0,4885.66
Казань,16,89007.0,19443.0,1126.0,5562.94
Красноярск,25,141345.0,24291.0,1113.0,5653.80
Москва,27,118194.0,19051.0,1479.0,4377.56
Новосибирск,29,170481.0,28251.0,0.0,5878.66
Омск,15,77763.0,17561.0,2263.0,5184.20
Санкт-Петербург,28,146244.0,23703.0,1309.0,5223.00
Томск,38,218112.0,25467.0,1322.0,5739.79


## 31. Анализ причин визитов

In [71]:
visits_clean.columns

Index(['visit_id', 'cat_id', 'visit_date', 'reason', 'clinic', 'cost_rub',
       'follow_up_days'],
      dtype='str')

In [72]:
cats_reason_analytics = (
    visits_clean
    .groupby('reason')
    .agg(
        visits_count = ('visit_id', 'count'),
        total_cost = ('cost_rub','sum'),
        avg_cost = ('cost_rub','mean')
    )
).round(2)

cats_reason_analytics

,visits_count,total_cost,avg_cost
reason,,,
анализ крови,50,276276.0,5638.29
вакцинация,20,105640.0,5282.00
дерматология,40,218268.0,5596.62
контроль веса,40,205545.0,5409.08
профилактический осмотр,40,198331.0,5360.30
стоматология,20,113427.0,5969.84


## 32. Работа с временными данными

In [73]:
visits_clean['visit_year'] = visits_clean['visit_date'].dt.year
visits_clean['visit_month'] = visits_clean['visit_date'].dt.to_period('M')

In [74]:
visits_clean.head(5)

,visit_id,cat_id,visit_date,reason,clinic,cost_rub,follow_up_days,visit_year,visit_month
0,VIS0001,CAT001,2026-08-09,профилактический осмотр,Айболит,NaN,14.0,2026,2026-08
1,VIS0002,CAT001,2026-06-17,анализ крови,Белый клык,1641.0,NaN,2026,2026-06
2,VIS0003,CAT001,2026-04-25,дерматология,Доктор Вет,2182.0,NaN,2026,2026-04
3,VIS0004,CAT002,2026-07-23,вакцинация,Кот и пёс,1479.0,NaN,2026,2026-07
4,VIS0005,CAT003,2026-07-06,анализ крови,Белый клык,1858.0,NaN,2026,2026-07


In [75]:
monthly_visits = (
    visits_clean
    .groupby('visit_month')
    .agg(
        visits = ('visit_id', 'count'),
        total_cost = ('cost_rub', 'sum')
    )
)

monthly_visits

,visits,total_cost
visit_month,,
2025-06,9,37012.0
2025-07,14,65186.0
2025-08,13,73927.0
2025-09,14,102932.0
2025-10,18,119373.0
2025-11,15,113361.0
2025-12,15,110825.0
2026-01,14,89163.0
2026-02,16,81661.0


# Часть IV. Полезные практические приёмы

## 33. rename()

In [76]:
example = cats_clean.rename(
    columns={
        "cat_id": "id",
        'weight_kg': 'weight'
    }
)
example[['id', 'name', 'weight']].head()

,id,name,weight
0,CAT001,Лео,8.4
1,CAT002,Лео,4.4
2,CAT003,Злата,6.3
3,CAT004,Мия,5.2
4,CAT005,Кира,5.15


## 34. assign()

In [78]:
(
    cats_clean
    .assign(
        weight_g = lambda df: df['weight_kg'] * 1000,
        height_m = lambda df: df['height_cm'] / 100
    )
    [['name', 'weight_g', 'height_m']]
    .head()
)

,name,weight_g,height_m
0,Лео,8400.0,0.37
1,Лео,4400.0,0.31
2,Злата,6300.0,0.35
3,Мия,5200.0,0.33
4,Кира,5150.0,0.29


## 35. method chaining

In [80]:
moscow_stats = (
    cats_clean
    .query("city == 'Москва'")
    .dropna(subset='weight_kg')
    .groupby('breed_key')
    .agg(
        cats = ('cat_id', 'count'),
        avg_weight = ('weight_kg', 'mean')
    )
    .sort_values('cats', ascending=False)
    .round(2)
)

moscow_stats

,cats,avg_weight
breed_key,,
абиссинская,3,3.73
британская короткошёрстная,3,4.85
канадский сфинкс,3,4.2
норвежская лесная,3,6.7
персидская,3,4.85
сибирская,2,6.6
русская голубая,1,6.1
рэгдолл,1,6.1
шотландская вислоухая,1,6.3


## 36. Проверки качества данных

In [83]:
assert cats_clean['cat_id'].isna().sum() == 0
assert cats_clean['cat_id'].nunique() == len(cats_clean)
assert cats_clean['sex'].isin(['кот', 'кошка']).all()

print('Базовые проверки пройдены')

Базовые проверки пройдены


### Проверка диапазонов

In [84]:
cats_clean['weight_kg'].describe().round(2)

count    120.0
mean      5.37
std       1.41
min        2.9
25%       4.47
50%       5.15
75%        6.1
max        9.2
Name: weight_kg, dtype: Float64

In [87]:
cats_clean[
    (cats_clean['weight_kg'] < 1) |
    (cats_clean['weight_kg'] > 15)
]

,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,sterilized,owner_id,weight_g,is_large,sex_code,weight_category,height_m,breed_key


## 37. Экспорт результата

In [90]:
OUTPUT_CSV = DATA_DIR / 'cats_clean.csv'
OUTPUT_XLSX = DATA_DIR / 'cats_analytics.xlsx'

cats_clean.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding='utf-8'
)

cats_analytics.to_excel(
    OUTPUT_XLSX,
    index=False
)

# Часть V. Итоговый pipeline

Ниже — компактная версия полного сценария, которую можно адаптировать под реальные проекты.


In [91]:
# 1. Загрузка
cats_pipeline = pd.read_csv(CSV_PATH)

# 2. Удаление дублей
cats_pipeline = cats_pipeline.drop_duplicates().copy()

# 3. Текстовые поля
cats_pipeline["breed_key"] = (
    cats_pipeline["breed"]
    .str.strip()
    .str.lower()
)

for col in ["vaccinated", "sterilized"]:
    cats_pipeline[col] = (
        cats_pipeline[col]
        .str.strip()
        .str.lower()
    )

# 4. Вес
cats_pipeline["weight_kg"] = pd.to_numeric(
    cats_pipeline["weight_kg"]
    .astype("string")
    .str.strip()
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

# 5. Даты
cats_pipeline["birth_date"] = pd.to_datetime(
    cats_pipeline["birth_date"],
    errors="coerce",
    dayfirst=True
)

# 6. Справочник пород
breeds_pipeline = pd.read_excel(
    XLSX_PATH,
    sheet_name="breeds"
)

breeds_pipeline["breed_key"] = (
    breeds_pipeline["breed"]
    .str.strip()
    .str.lower()
)

cats_pipeline = cats_pipeline.merge(
    breeds_pipeline,
    on="breed_key",
    how="left",
    validate="many_to_one",
    suffixes=("", "_ref")
)

# 7. Визиты
visits_pipeline = pd.read_excel(
    XLSX_PATH,
    sheet_name="visits"
)

visits_pipeline["visit_date"] = pd.to_datetime(
    visits_pipeline["visit_date"],
    errors="coerce"
)

visit_stats = (
    visits_pipeline
    .groupby("cat_id")
    .agg(
        visits_count=("visit_id", "count"),
        total_cost=("cost_rub", "sum"),
        last_visit=("visit_date", "max")
    )
    .reset_index()
)

cats_pipeline = cats_pipeline.merge(
    visit_stats,
    on="cat_id",
    how="left",
    validate="one_to_one"
)

cats_pipeline["visits_count"] = (
    cats_pipeline["visits_count"]
    .fillna(0)
    .astype(int)
)

cats_pipeline["total_cost"] = (
    cats_pipeline["total_cost"]
    .fillna(0)
)

# 8. Итог
cats_pipeline.head()

/var/folders/24/z61d72zx27z5ml8twybhwmm40000gn/T/ipykernel_2471/2870887213.py:31: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  cats_pipeline["birth_date"] = pd.to_datetime(


,cat_id,name,breed,sex,color,birth_date,weight_kg,height_cm,city,vaccinated,...,breed_key,breed_ref,origin,coat_type,typical_weight_min_kg,typical_weight_max_kg,typical_lifespan_years,visits_count,total_cost,last_visit
0,CAT001,Лео,Мейн-кун,кот,рыжий,2021-11-24,8.4,37.0,Томск,да,...,мейн-кун,Мейн-кун,США,полудлинная,5.0,9.0,13,3,3823.0,2026-08-09
1,CAT002,Лео,Британская короткошёрстная,кошка,рыжий,2016-09-04,4.4,31.0,Москва,да,...,британская короткошёрстная,Британская короткошёрстная,Великобритания,короткая,4.0,8.0,14,1,1479.0,2026-07-23
2,CAT003,Злата,Шотландская вислоухая,кот,табби,2024-04-01,6.3,35.0,Москва,да,...,шотландская вислоухая,Шотландская вислоухая,Великобритания,короткая,3.0,6.0,14,1,1858.0,2026-07-06
3,CAT004,Мия,Сибирская,кошка,табби,2024-09-06,5.2,33.0,Санкт-Петербург,нет,...,сибирская,Сибирская,Россия,полудлинная,4.0,8.0,15,2,5015.0,2026-06-19
4,CAT005,Кира,Сиамская,кот,серый,2023-09-27,<NA>,29.0,Екатеринбург,нет,...,сиамская,Сиамская,Таиланд,короткая,3.0,5.5,16,3,9471.0,2026-06-02


## Итоговая аналитическая таблица

In [93]:
final_report = (
    cats_pipeline
    .groupby("city")
    .agg(
        cats=("cat_id", "count"),
        avg_weight=("weight_kg", "mean"),
        visits=("visits_count", "sum"),
        total_vet_cost=("total_cost", "sum")
    )
    .sort_values("cats", ascending=False)
    .round(2)
)

final_report

,cats,avg_weight,visits,total_vet_cost
city,,,,
Москва,20,5.24,27,118194.0
Томск,19,5.6,38,218112.0
Екатеринбург,16,5.98,32,156341.0
Новосибирск,16,5.46,29,170481.0
Санкт-Петербург,15,5.06,28,146244.0
Красноярск,13,5.3,25,141345.0
Казань,12,4.72,16,89007.0
Омск,9,5.64,15,77763.0


## Что изучено

В этом ноутбуке показаны:

- `read_csv()` и `read_excel()`;
- `Series`, `DataFrame`, `Index`;
- `head()`, `tail()`, `sample()`, `info()`, `describe()`;
- выбор столбцов, `loc`, `iloc`;
- фильтрация, `isin()`, `query()`;
- сортировка;
- строковые методы `.str`;
- пропуски и дубликаты;
- `to_numeric()` и `to_datetime()`;
- создание новых признаков;
- `map()` и `apply()`;
- `value_counts()`;
- `groupby()` и `agg()`;
- `pivot_table()`;
- `merge()` для `many-to-one` и `one-to-many`;
- анализ временных данных;
- method chaining;
- проверки качества;
- экспорт результатов.

Ноутбук можно использовать как основу для практического занятия или лабораторной работы.
